### 0: Downloading Programmes

In [ ]:
import gurobipy as gp
from gurobipy import GRB
import pyomo.environ as pyo
from pyomo.opt import SolverFactory
import pandas as pd
from pathlib import Path
from collections import defaultdict


### 1: Dataframes

In [ ]:
base = Path("output_folder")

df_resources = pd.read_csv(base / "resources.csv") # resource ID and resource type
df_constraints = pd.read_csv(base / "constraints.csv") # constraint ID and type, Name, required?, weight and cost function

df_times = pd.read_csv(base / "times.csv").sort_values("order_index") # timeslots and their timegroups (days)


# df_events = event with its duration, course, groups, role, classes and teacher)

events = pd.read_csv(base / "events.csv") 
event_groups = pd.read_csv(base / "event_eventgroup_membership.csv")
event_resources = pd.read_csv(base / "event_resources.csv")

# Events each group is part of
groups_per_event = (
    event_groups
    .groupby("event_id")["event_group_id"]
    .apply(list)
    .reset_index(name="groups")
)

# Teacher of each event
teachers_per_event = event_resources[event_resources["role"] == "Teacher"]
teachers_per_event = ( teachers_per_event[["event_id", "reference_resource_id"]]
    .groupby("event_id")["reference_resource_id"]
    .apply(list)
    .reset_index(name="teachers")
) 

#Classes in each event
classes_per_event = event_resources[event_resources["role"].astype(str).str.startswith("Class")]
classes_per_event = ( classes_per_event[["event_id", "reference_resource_id"]]
    .groupby("event_id")["reference_resource_id"]
    .apply(list)
    .reset_index(name="classes")
)

# Role of each event
role_per_event = event_resources[event_resources["resource_type_ref"] == "Room"]
role_per_event = (role_per_event[["event_id", "role"]]
    .groupby("event_id")["role"]
    .first()
    .reset_index(name="role")
)


# Room of each event
room_per_event = event_resources[event_resources["role"] == "Room"]
room_per_event = (room_per_event[["event_id", "reference_resource_id"]]
    .groupby("event_id")["reference_resource_id"]
    .apply(list)
    .reset_index(name="room")
)

# Students of each event
students_per_event = event_resources[event_resources["role"] == "Student"]
students_per_event = (students_per_event[["event_id", "reference_resource_id"]]
    .groupby("event_id")["reference_resource_id"]
    .apply(list)
    .reset_index(name="students")
)

df_events = (
    events.rename(columns={"course_ref": "course_reference"})[["event_id", "duration", "course_reference"]]
    .merge(groups_per_event, on="event_id", how="left")
    .merge(role_per_event, on="event_id", how="left")
    .merge(room_per_event, on="event_id", how="left")
    .merge(classes_per_event, on="event_id", how="left")
    .merge(teachers_per_event, on="event_id", how="left")
    .merge(students_per_event, on="event_id", how="left")
) # event with its duration, course, groups, role, classes and teacher)


df_events["course_reference"] = (
    df_events["groups"]
    .explode()
    .where(lambda s: s.astype(str).str.startswith("Course"))
    .groupby(level=0)
    .first()
)



resource_groups = pd.read_csv(base / "resource_group_membership.csv")

# Keep only what we need from memberships and aggregate group memberships
groups_per_resource = (
    resource_groups
    .groupby("resource_id")["resource_group_id"]
    .apply(list)
    .reset_index(name="resource_groups")
)

# Merge onto resources
df_res_with_groups = (
    df_resources[["resource_id", "resource_type_ref"]]
    .merge(groups_per_resource, on="resource_id", how="left")
)

# Replace NaN with empty list (resources that have no group membership)
df_res_with_groups["resource_groups"] = df_res_with_groups["resource_groups"].apply(
    lambda x: x if isinstance(x, list) else []
)


# Split into separate tables
df_teachers = df_res_with_groups[df_res_with_groups["resource_type_ref"] == "Teacher"].reset_index(drop=True) # teacher id, resource type and list of groups
df_classes  = df_res_with_groups[df_res_with_groups["resource_type_ref"] == "Class"].reset_index(drop=True) # class id, resource type and list of groups
df_rooms    = df_res_with_groups[df_res_with_groups["resource_type_ref"] == "Room"].reset_index(drop=True) # room id, resource type and list of groups
df_students = df_res_with_groups[df_res_with_groups["resource_type_ref"] == "Student"].reset_index(drop=True) # room id, resource type and list of groups


df_res_long = (
    df_resources[["resource_id", "resource_type_ref"]]
    .merge(resource_groups[["resource_id", "resource_group_id"]], on="resource_id", how="left")
    .rename(columns={"resource_group_id": "resource_group"})
)
df_teachers_long = df_res_long[df_res_long["resource_type_ref"] == "Teacher"] # teacher id, resource type and seperate groups
df_classes_long  = df_res_long[df_res_long["resource_type_ref"] == "Class"] # class id, resource type and seperate groups
df_rooms_long    = df_res_long[df_res_long["resource_type_ref"] == "Room"] # room id, resource type and seperate groups
df_students_long = df_res_long[df_res_long["resource_type_ref"] == "Student"].reset_index(drop=True) # room id, resource type and list of groups



df_constraint_applied = pd.read_csv(base / "constraint_applies_to.csv")
df_constraint_params = pd.read_csv(base / "constraint_params.csv")

df_constraint_params = df_constraint_params.merge(
    df_constraints[["constraint_id", "constraint_type"]],
    on="constraint_id",
    how="left"
)

df_constraint_params = df_constraint_params.merge(
    df_constraint_applied[["constraint_id", "reference"]],
    on="constraint_id",
    how="left"
)

In [ ]:
#print(df_resources)
#print(df_constraints)

#print(df_times)
print(df_events)
#print(df_rooms)
#print(df_teachers)
#print(df_classes)

#df_constraint_params



### 2a: Model setup and helpers


1) pre-assigned times colum (from events.csv)
2) workload colum which takes workload value from event_resources.csv and if that doesnt exist it takes duration value from events.csv
3) A new teachers_needed column should state the number of teachers that need to be assigned, which can be counted from event_resources.csv

In [ ]:
def as_list(x):
    if x is None:
        return []
    if isinstance(x, (list, tuple, set)):
        return list(x)
    return [x]

In [ ]:
### Sets

E = df_events["event_id"].tolist() # All events
T = df_times["time_id"].tolist() # All timeslots
R = df_rooms["resource_id"].tolist() # All rooms

all_teachers = df_teachers["resource_id"].tolist()
all_classes = df_classes["resource_id"].tolist()
all_students = df_students["resource_id"].tolist()


# event -> duration/teachers/classes (lists)
event_to_duration = dict(zip(df_events["event_id"], df_events["duration"].astype(int))) # duration of each event
event_to_teachers = dict(zip(df_events["event_id"], df_events["teachers"])) # teachers for each event
event_to_classes  = dict(zip(df_events["event_id"], df_events["classes"])) # classes for each event
#event_to_room = dict(zip(df_events["event_id"], df_events["room"])) # Room for each event
event_to_students  = dict(zip(df_events["event_id"], df_events["students"])) # students for each event

#event -> room group of each event
#event_to_roomgroup = dict(zip(df_events["event_id"], df_events["role"])) 

room_to_group = (
    df_rooms_long[["resource_id", "resource_group"]]
    .dropna(subset=["resource_group"])
    .drop_duplicates()
    .groupby("resource_id")["resource_group"]
    .apply(list)
    .to_dict()
)

group_to_room = (
    df_rooms_long[["resource_id", "resource_group"]]
    .dropna(subset=["resource_group"])
    .drop_duplicates()
    .groupby("resource_group")["resource_id"]
    .apply(list)
    .to_dict()
)


T_index = {t:i for i,t in enumerate(T)}

# Keep only true unavailability constraints (exclude "NotPreferred")
df_unavail_times = df_constraint_params[
    (df_constraint_params["constraint_type"] == "AvoidUnavailableTimesConstraint") &
    (df_constraint_params["path"] == "Times/Time") &
    (~df_constraint_params["constraint_id"].astype(str).str.contains("NotPreferred", case=False, na=False))
].copy()[["reference", "value"]]

# resource -> set(unavailable times)
resource_unavailability = (
    df_unavail_times
    .groupby("reference")["value"]
    .agg(lambda s: set(s))
    .to_dict()
)

In [ ]:
### Feasible Start times for events:

#T_index = {t:i for i,t in enumerate(T)}

time_to_day = dict(zip(df_times["time_id"], df_times["day_ref"]))

# Build feasible starts per event (no crossing day boundary) (implicitely covers constraints 5,9,10,11,12)
feasible_starts = {}
for e in E:
    d = int(event_to_duration[e]) # number of periods needed
    starts = []
    for ts in T: # for all possible start times
        i0 = T_index[ts]
        # must have enough periods remaining in the global list
        if i0 + d - 1 >= len(T):
            continue
        day0 = time_to_day[ts]
        block = T[i0:i0 + d]
        # enforce all times in the block are same day
        if all(time_to_day[t] == day0 for t in block):
            starts.append(ts)
    feasible_starts[e] = starts

# If event e starts at ts, then occ_times[(e, ts)] = [t1, t2, ...] is the list of all times occupied by that event
occ_times = {}
for e in E:
    d = int(event_to_duration[e])
    for ts in feasible_starts[e]:
        i0 = T_index[ts]
        occ_times[(e, ts)] = T[i0:i0 + d]


# Helper: all (e, ts) pairs that occupy a given time t
# ie for Fr_3, if event 1 is duration 1 and event 2 is duration 2, then (1, Fr_3) and (2, Fr_2) and (2, Fr_3) occupy it
occupies_at_time = {t: [] for t in T}
for e in E:
    for ts in feasible_starts[e]:
        for t in occ_times[(e, ts)]:
            occupies_at_time[t].append((e, ts))



In [ ]:

# Filter feasible starts against both teacher and class unavailability
feasible_starts_filtered = {}

for e in E:
    starts_ok = []
    teachers = set(as_list(event_to_teachers[e]))
    classes  = set(as_list(event_to_classes[e]))
    resources_for_event = teachers | classes

    for ts in feasible_starts[e]:
        occ = occ_times[(e, ts)]
        violates = False

        for res in resources_for_event:
            unavail = resource_unavailability.get(res, set())
            if any(t in unavail for t in occ):
                violates = True
                break

        if not violates:
            starts_ok.append(ts)

    feasible_starts_filtered[e] = starts_ok

# Replace
feasible_starts = feasible_starts_filtered
#feasible_starts


# rebuild occ_times
occ_times = {}
for e in E:
    d = int(event_to_duration[e])
    for ts in feasible_starts[e]:
        i0 = T_index[ts]
        occ_times[(e, ts)] = T[i0:i0 + d]

# rebuild occupies_at_time
occupies_at_time = {t: [] for t in T}
for e in E:
    for ts in feasible_starts[e]:
        for t in occ_times[(e, ts)]:
            occupies_at_time[t].append((e, ts))

#feasible_starts

In [ ]:
### Feasible Rooms

# all rooms
all_rooms = set(df_rooms["resource_id"].astype(str))

# only prefer-resource constraints
pref_cids = set(df_constraints.loc[
    df_constraints["constraint_type"] == "PreferResourcesConstraint", "constraint_id"
])

# keep only room-role constraints
cp = df_constraint_params[["constraint_id", "path", "value"]].drop_duplicates()
role = cp[cp["path"] == "Role"].set_index("constraint_id")["value"].astype(str)
#pref_cids = {cid for cid in pref_cids if role.get(cid, "") == "Room"}

# room group -> rooms
rg_to_rooms = (
    resource_groups.groupby("resource_group_id")["resource_id"]
    .apply(lambda s: set(s.astype(str)) & all_rooms).to_dict()
)

# constraint -> preferred rooms
cid_to_rooms = {}
for cid in pref_cids:
    rooms = set(cp[(cp["constraint_id"] == cid) & (cp["path"] == "Resources/Resource")]["value"].astype(str)) & all_rooms
    rgs = set(cp[(cp["constraint_id"] == cid) & (cp["path"] == "ResourceGroups/ResourceGroup")]["value"].astype(str))
    for rg in rgs:
        rooms |= rg_to_rooms.get(rg, set())
    cid_to_rooms[cid] = rooms

# event_group -> events
eg_to_events = event_groups.groupby("event_group_id")["event_id"].apply(lambda s: set(s.astype(str))).to_dict()


# event -> feasible rooms
feasible_rooms = {str(e): set(all_rooms) for e in E}
event_is_restricted = {str(e): False for e in E}

for cid in pref_cids:
    evs = set(df_constraint_applied[
        (df_constraint_applied["constraint_id"] == cid) &
        (df_constraint_applied["path"] == "AppliesTo/Events/Event")
    ]["reference"].astype(str))

    egs = set(df_constraint_applied[
        (df_constraint_applied["constraint_id"] == cid) &
        (df_constraint_applied["path"] == "AppliesTo/EventGroups/EventGroup")
    ]["reference"].astype(str))

    for eg in egs:
        evs |= eg_to_events.get(eg, set())

    for e in evs:
        if e in feasible_rooms:
            if not event_is_restricted[e]:
                feasible_rooms[e] = set()
                event_is_restricted[e] = True
            feasible_rooms[e] |= cid_to_rooms.get(cid, set())

# sort for stable output
feasible_rooms = {e: sorted(v) for e, v in feasible_rooms.items()}

#feasible_rooms

### 2b: Pyomo Model part 1: Sets and Decision Variables

Constraints:

Part 1: Event -> time (hard constraints):

1) AssignTimes: [gr_EventsGeneral] Every event must be assigned a time
2) SpreadEventsClassDayConflict: events from each course have a maximum and minimum specified number of events in each time group [monday, tuesday etcc...] (here max 1 per day) 
3) AvoidClashes: No clashes for any of the resources
4) AvoidUnavailable: Select teachers cannot teach at given times
5) ClusterBusyRequiredDaysOff2: Teacher 18 must have between min and max busy days per week 

Extra: Number of events at any time should be less than the number of total rooms 

6) PreferResource_number: Select Event will prefer to be in one of the list of given rooms - implement this as "number of events of a certain group at and time should be less than the total number of that groups preferred rooms


Part 2: Event -> time (add soft constraints):

1) SpreadEventsNeighbordays: each course can have a maximum and minimum specified number of events in each time group (here neighbouring day time groups) (soft: weight=8, linear)
2) ClassWeekStability_number: each course can have a maximum and minimum specified number of events in each time group (here week 1 and week 2) (soft: weight=8, linear) (soft: weight=8, linear)
3) LimitIdleStudents: [all_students] students should have a minimum of 0 and maximum of 0 idle timeperiods (no timeperiod where there is no event but there is an earlier and later event on the same day) on every given time group (day) (soft: weight = 7, linear) 
4) LimitIdleTeachers: [gr_teachers] teachers should have a minimum of 0 and maximum of 0 idle timeperiods (no timeperiod where there is no event but there is an earlier and later event on the same day) on every given time group (day) (soft: weight = 6, linear)
5) ClusterBusyDaysOccupiedPenalty: Teachers should have zero busy days (ie as few busy days as possible) (soft: weight=1, linear)
6) ClusterBusyDaysOffStudents: Students should have 10 busy days (at least one event every day) (soft: weight=1, linear)
7) LimitBusyMoreThanOne: If a resource is active on a day, it should have between min and max busy periods in that day (soft: weight=4, linear )

Part 3: Event -> room:

1) Every event must be assigned exactly 1 room
2) At most one event can occupy a room at a time


### MODEL

This is a 2-stage model where we solve for rooms and times at the same time (feasibility problem first)

In [ ]:
from collections import defaultdict
import pyomo.environ as pyo

# ---- Model ----
m1 = pyo.ConcreteModel()

# ---- Sets ----
m1.E = pyo.Set(initialize=E)
m1.T = pyo.Set(initialize=T, ordered=True)
m1.R = pyo.Set(initialize=R)

m1.Teachers = pyo.Set(initialize=all_teachers)
m1.Classes = pyo.Set(initialize=all_classes)

# ---- Feasible (event, start, room) triples ----
m1.X_index = pyo.Set(
    dimen=3,
    initialize=[
        (e, ts, r)
        for e in E
        for ts in feasible_starts[e]
        for r in feasible_rooms[e]
    ]
)

# Map each occupied timeslot to the feasible (event, start, room) triples that use it
occupies_at_time = defaultdict(list)
occupies_at_time_room = defaultdict(list)

for (e, ts, r) in m1.X_index:
    for t in occ_times[(e, ts)]:
        occupies_at_time[t].append((e, ts, r))
        occupies_at_time_room[(t, r)].append((e, ts, r))

# ---- Decision Variables ----
# x[e, ts, r] = 1 if event e starts at ts in room r
m1.x = pyo.Var(m1.X_index, domain=pyo.Binary)


In [ ]:
# Hard

# 1) Each event scheduled exactly once (one start time + one room)
def one_start_room_rule(m, e):
    return sum(
        m.x[e, ts, r]
        for ts in feasible_starts[e]
        for r in feasible_rooms[e]
        if (e, ts, r) in m.X_index
    ) == 1

m1.OneStartRoom = pyo.Constraint(m1.E, rule=one_start_room_rule)

In [ ]:
# 2) No room conflicts: at most one event can occupy a room at a time
def room_conflict_rule(m, r, t):
    terms = [
        m.x[e, ts, r]
        for (e, ts, r) in occupies_at_time_room.get((t, r), [])
    ]
    if not terms:
        return pyo.Constraint.Skip
    return sum(terms) <= 1

m1.RoomConflict = pyo.Constraint(m1.R, m1.T, rule=room_conflict_rule)

In [ ]:
# 3) No teacher conflicts: a teacher can teach at most one event at a time
def teacher_conflict_rule(m, teacher, t):
    terms = [
        m.x[e, ts, r]
        for (e, ts, r) in occupies_at_time.get(t, [])
        if teacher in as_list(event_to_teachers[e])
    ]
    if not terms:
        return pyo.Constraint.Skip
    return sum(terms) <= 1

m1.TeacherConflict = pyo.Constraint(m1.Teachers, m1.T, rule=teacher_conflict_rule)

In [ ]:
# 4) No student conflicts: a student can attend at most one event at a time
def student_conflict_rule(m, stu, t):
    terms = [
        m.x[e, ts, r]
        for (e, ts, r) in occupies_at_time.get(t, [])
        if stu in as_list(event_to_students[e])
    ]
    if not terms:
        return pyo.Constraint.Skip
    return sum(terms) <= 1

m1.StudentConflict = pyo.Constraint(all_students, m1.T, rule=student_conflict_rule)

In [ ]:
from collections import defaultdict

# 1) Map event -> course_reference
event_to_course = dict(zip(df_events["event_id"], df_events["course_reference"]))

# 2) Build course set
courses = sorted(set(event_to_course[e] for e in E))

# 3) Days from df_times
days = sorted(df_times["day_ref"].dropna().unique().tolist())

# 4) Collect relevant x vars by (course, day)
#    key -> list of feasible (e, ts, r) triples for that course/day
course_day_terms = defaultdict(list)

for (e, ts, r) in m1.X_index:
    c = event_to_course[e]
    d = time_to_day[ts]
    course_day_terms[(c, d)].append((e, ts, r))

# 5) Define Pyomo sets
m1.COURSES = pyo.Set(initialize=courses)
if not hasattr(m1, "DAYS"):
    m1.DAYS = pyo.Set(initialize=days)

# 6) At most one event from the same course per day
def event_spread_max_rule(m, c, d):
    terms = course_day_terms.get((c, d), [])
    if not terms:
        return pyo.Constraint.Skip
    return sum(m.x[e, ts, r] for (e, ts, r) in terms) <= 1

m1.EventSpreadMax = pyo.Constraint(m1.COURSES, m1.DAYS, rule=event_spread_max_rule)


In [ ]:
# -------- data from the constraint --------
cid = "ClusterBusyRequiredDaysOff2"
teacher_id = "Teacher18"

min_busy = int(df_constraint_params.loc[
    (df_constraint_params["constraint_id"] == cid) &
    (df_constraint_params["path"] == "Minimum"),
    "value"
].iloc[0])

max_busy = int(df_constraint_params.loc[
    (df_constraint_params["constraint_id"] == cid) &
    (df_constraint_params["path"] == "Maximum"),
    "value"
].iloc[0])

# -------- sets/vars --------
m1.CBD_DAYS = pyo.Set(initialize=days)
m1.teacher_busy_day = pyo.Var(m1.CBD_DAYS, domain=pyo.Binary)  # 1 if teacher busy that day

# precompute teacher terms by day
teacher_day_terms = {}
for d in days:
    teacher_day_terms[d] = [
        (e, ts, r)
        for (e, ts, r) in m1.X_index
        if (teacher_id in as_list(event_to_teachers.get(e)))
        and (time_to_day[ts] == d)
    ]

M_day = {d: len(teacher_day_terms[d]) for d in days}

# if teacher has any event in day d => teacher_busy_day[d] = 1
def cbd_link_upper_rule(m, d):
    terms = teacher_day_terms[d]
    if not terms:
        return m.teacher_busy_day[d] == 0
    return sum(m.x[e, ts, r] for (e, ts, r) in terms) <= M_day[d] * m.teacher_busy_day[d]

def cbd_link_lower_rule(m, d):
    terms = teacher_day_terms[d]
    if not terms:
        return pyo.Constraint.Skip
    return m.teacher_busy_day[d] <= sum(m.x[e, ts, r] for (e, ts, r) in terms)

m1.CBD_LinkUpper = pyo.Constraint(m1.CBD_DAYS, rule=cbd_link_upper_rule)
m1.CBD_LinkLower = pyo.Constraint(m1.CBD_DAYS, rule=cbd_link_lower_rule)

# hard bounds: min <= number of busy days <= max
m1.CBD_MinBusy = pyo.Constraint(
    expr=sum(m1.teacher_busy_day[d] for d in m1.CBD_DAYS) >= min_busy
)
m1.CBD_MaxBusy = pyo.Constraint(
    expr=sum(m1.teacher_busy_day[d] for d in m1.CBD_DAYS) <= max_busy
)


In [ ]:
if hasattr(m1, "PreferResourceGroupCapacity"):
    m1.del_component(m1.PreferResourceGroupCapacity)

m1.PreferResourceGroupCapacity = pyo.ConstraintList()

# prefer-resource constraints only
pref_cids = set(
    df_constraints.loc[
        df_constraints["constraint_type"] == "PreferResourcesConstraint",
        "constraint_id"
    ]
)

# constraint -> events it applies to
cid_to_events = (
    df_constraint_applied.loc[
        (df_constraint_applied["constraint_id"].isin(pref_cids)) &
        (df_constraint_applied["path"] == "AppliesTo/Events/Event"),
        ["constraint_id", "reference"]
    ]
    .groupby("constraint_id")["reference"]
    .apply(lambda s: set(s.astype(str).str.strip()))
    .to_dict()
)

# optional: include events applied via event groups if needed
eg_to_events = (
    event_groups.groupby("event_group_id")["event_id"]
    .apply(lambda s: set(s.astype(str).str.strip()))
    .to_dict()
)

# constraint -> preferred rooms from explicit resources
cp = df_constraint_params[["constraint_id", "path", "value"]].drop_duplicates().copy()

cid_to_pref_rooms = (
    cp.loc[
        (cp["constraint_id"].isin(pref_cids)) &
        (cp["path"] == "Resources/Resource"),
        ["constraint_id", "value"]
    ]
    .groupby("constraint_id")["value"]
    .apply(lambda s: set(s.astype(str).str.strip()))
    .to_dict()
)

for cid in pref_cids:
    applies_events = set(cid_to_events.get(cid, set()))
    pref_rooms = set(cid_to_pref_rooms.get(cid, set()))
    n_rooms = len(pref_rooms)

    if not applies_events or n_rooms == 0:
        continue

    for t in m1.T:
        terms = [
            m1.x[e, ts, r]
            for (e, ts, r) in occupies_at_time.get(t, [])
            if e in applies_events
        ]
        if terms:
            m1.PreferResourceGroupCapacity.add(sum(terms) <= n_rooms)


In [ ]:
# ---- Objective ---- (hard constraints only)
m1.obj = pyo.Objective(expr=0, sense=pyo.minimize)

In [ ]:
import numpy as np

In [ ]:
print("Pyomo variables:", m1.nvariables())
print("Pyomo constraints:", m1.nconstraints())


In [ ]:
seeds = [10]

solve_times = []
objectives = []
num_vars = []
num_cons = []
solve_status = []

start_x_list = []
chosen_assignments = []

import time as time_module

def clear_all_var_values(model):
    for v in model.component_data_objects(pyo.Var, active=True):
        v.set_value(None, skip_validation=True)

for seed in seeds:
    clear_all_var_values(m1)

    #solver = pyo.SolverFactory("gurobi")
    solver = pyo.SolverFactory("gurobi_direct")
    solver.options["TimeLimit"] = 300
    solver.options["Seed"] = seed
    solver.options["Threads"] = 1

    start_time = time_module.time()
    res = solver.solve(m1, tee=False, load_solutions=False)
    elapsed_time = time_module.time() - start_time

    status = str(res.solver.status)
    term = str(res.solver.termination_condition)
    has_solution = len(res.solution) > 0

    if has_solution:
        m1.solutions.load_from(res)

    solve_times.append(elapsed_time)
    num_vars.append(m1.nvariables())
    num_cons.append(m1.nconstraints())
    solve_status.append((status, term))

    if has_solution:
        try:
            obj_val = pyo.value(m1.obj)
        except:
            obj_val = float("nan")
            has_solution = False
    else:
        obj_val = float("nan")

    objectives.append(obj_val)

    # Store full solution: (e, ts, r) -> value
    seed_solution = {}
    for (e, ts, r) in m1.X_index:
        val = pyo.value(m1.x[e, ts, r], exception=False)
        seed_solution[(e, ts, r)] = 0.0 if val is None else float(val)
    start_x_list.append(seed_solution)

    # Store compact chosen assignment: event -> time, room
    chosen_map = {}
    if has_solution:
        for e in m1.E:
            picked = [
                (ts, r)
                for ts in feasible_starts[e]
                for r in feasible_rooms[e]
                if (e, ts, r) in m1.X_index
                and pyo.value(m1.x[e, ts, r], exception=False) is not None
                and pyo.value(m1.x[e, ts, r], exception=False) > 0.5
            ]

            if len(picked) == 1:
                ts, r = picked[0]
                chosen_map[e] = {"time": ts, "room": r}
            elif len(picked) == 0:
                chosen_map[e] = {"time": None, "room": None}
            else:
                raise ValueError(f"Event {e}: expected 1 assignment, got {picked}")
    else:
        for e in m1.E:
            chosen_map[e] = {"time": None, "room": None}

    chosen_assignments.append(chosen_map)

# Keep compatibility with later cells
start_x = start_x_list[0]
chosen_start_room = chosen_assignments[0]
res = res

valid_objs = [v for v in objectives if not np.isnan(v)]

avg_time = sum(solve_times) / len(solve_times)
avg_vars = sum(num_vars) / len(num_vars)
avg_cons = sum(num_cons) / len(num_cons)

print("=" * 50)
print("SOLVER METRICS SUMMARY - PHASE 1")
print("=" * 50)
print(f"Average Solve Time (s): {avg_time:.4f}")
print(f"Average Objective: {np.mean(valid_objs):.6f}" if valid_objs else "Average Objective: Not available")
print(f"Average Variables: {avg_vars:.0f}")
print(f"Average Constraints: {avg_cons:.0f}")
print(f"Statuses: {set(solve_status)}")
print("=" * 50)

print(f"\nPhase 1 produced {len(start_x_list)} runs (one per seed)")
for i in range(len(seeds)):
    assigned = sum(
        1 for e, rec in chosen_assignments[i].items()
        if rec["time"] is not None and rec["room"] is not None
    )
    obj_str = f"{objectives[i]:.6f}" if not np.isnan(objectives[i]) else "No solution"
    print(
        f"  Seed {seeds[i]}: assigned {assigned} events in {solve_times[i]:.2f}s, "
        f"Obj={obj_str}, Status={solve_status[i]}"
    )


### 2b: Model 1

This is the 3-stage model where we assign times first (feasibility stage then optimisation stage) and then rooms in a later model.

In [ ]:
# ---- Model ----
m1 = pyo.ConcreteModel()

# ---- Sets ----
m1.E = pyo.Set(initialize=E) # Set of events
m1.T = pyo.Set(initialize=T, ordered=True) # Set of timeslots (ordered)
m1.R = pyo.Set(initialize=R) # Set of rooms

m1.Teachers = pyo.Set(initialize=all_teachers) # set of all teachers
m1.Classes = pyo.Set(initialize=all_classes) # set of all classes
m1.S = pyo.Set(initialize=all_students)


# standard set up
m1.X_index = pyo.Set(
    dimen=2,
    initialize=[(e, ts) for e in E for ts in feasible_starts[e]]
)



# ---- Decision Variables ----
# Start-time decision variables:
# x[e, ts, r] = 1 if event e starts at time ts in room r
# Only define variables for feasible starts (and all rooms) (THIS SPEEDS UP MODEL)
m1.x = pyo.Var(m1.X_index, domain=pyo.Binary)

In [ ]:

# Standard set up

# 1) Each event scheduled exactly once (one start time)   (covers constraints 4)
def one_start_rule(m, e):
    return sum(m.x[e, ts] for ts in feasible_starts[e]) == 1
m1.OneStart = pyo.Constraint(m1.E, rule=one_start_rule)


# 3) No teacher conflicts: a teacher can teach at most one event at a time
def teacher_conflict_rule(m, teacher, t):
    terms = [
        m.x[e, ts]
        for (e, ts) in occupies_at_time.get(t, [])
        if teacher in as_list(event_to_teachers[e])
    ]
    if not terms:
        return pyo.Constraint.Skip   # or pyo.Constraint.Feasible
    return sum(terms) <= 1

m1.TeacherConflict = pyo.Constraint(all_teachers, m1.T, rule=teacher_conflict_rule)


#m1.ClassConflict = pyo.Constraint(all_classes, m1.T, rule=class_conflict_rule)

# 6) No student conflicts: a student can attend at most one event at a time
def student_conflict_rule(m, stu, t):
    terms = [
        m.x[e, ts]
        for (e, ts) in occupies_at_time.get(t, [])
        if stu in event_to_students[e]
    ]
    if not terms:
        return pyo.Constraint.Skip
    return sum(terms) <= 1

m1.StudentConflict = pyo.Constraint(all_students, m1.T, rule=student_conflict_rule)


# 5) Max number of simultaneous events cannot exceed number of rooms
m1.N_rooms = pyo.Param(initialize=len(R), within=pyo.NonNegativeIntegers)

def total_events_vs_rooms_rule(m, t):
    terms = [m.x[se, ts] for (se, ts) in occupies_at_time.get(t, [])]
    if not terms:
        return pyo.Constraint.Skip
    return sum(terms) <= m.N_rooms

m1.TotalEventsVsRooms = pyo.Constraint(m1.T, rule=total_events_vs_rooms_rule)



In [ ]:
### 3) Events that are part of the same course must be on different days (constraint 13)

# 1) Map event -> course_reference
event_to_course = dict(zip(df_events["event_id"], df_events["course_reference"]))

# 2) Build course groups (gr_C001 ... gr_C150)
courses = sorted(set(event_to_course[e] for e in E))

# 3) Days (time groups) from df_times (gr_Mo, gr_Tu, ...)
days = sorted(df_times["day_ref"].dropna().unique().tolist())

# 4) Collect relevant x vars by (course, day)
#    key -> list of all feasible (e, ts) combinations for given course and day
course_day_terms = defaultdict(list)

for (e, ts) in m1.X_index:
    c = event_to_course[e]
    d = time_to_day[ts]   # day of the start time
    course_day_terms[(c, d)].append((e, ts))


# 5) Define Pyomo sets for indexing
m1.COURSES = pyo.Set(initialize=courses)
m1.DAYS = pyo.Set(initialize=days)

# 6) Max-per-day constraint (Maximum = 1)
def event_spread_max_rule(m, c, d):
    terms = course_day_terms.get((c, d), [])
    if not terms:
        return pyo.Constraint.Skip
    return sum(m.x[e, ts] for (e, ts) in terms) <= 1 # at most one event for every course each day

m1.EventSpreadMax = pyo.Constraint(m1.COURSES, m1.DAYS, rule=event_spread_max_rule)



# Example from your data: min=0, max=1 for each day
#day_min = {d: 0 for d in days}
#day_max = {d: 1 for d in days}



#m.EventSpreadMin = pyo.Constraint(m.COURSES, m.DAYS, rule=event_spread_min_rule)
#m.EventSpreadMax = pyo.Constraint(m.COURSES, m.DAYS, rule=event_spread_max_rule)


In [ ]:

# -------- data from the constraint --------
cid = "ClusterBusyRequiredDaysOff2"
teacher_id = "Teacher18"

#day_groups = (
#    df_constraint_params.loc[
#        (df_constraint_params["constraint_id"] == cid) &
#        (df_constraint_params["path"] == "TimeGroups/TimeGroup"),
#        "value"
#    ]
#    .dropna().astype(str).tolist()
#)


min_busy = int(df_constraint_params.loc[
    (df_constraint_params["constraint_id"] == cid) &
    (df_constraint_params["path"] == "Minimum"),
    "value"
].iloc[0])

max_busy = int(df_constraint_params.loc[
    (df_constraint_params["constraint_id"] == cid) &
    (df_constraint_params["path"] == "Maximum"),
    "value"
].iloc[0])

# -------- sets/vars --------
m1.CBD_DAYS = pyo.Set(initialize=days)
m1.teacher_busy_day = pyo.Var(m1.CBD_DAYS, domain=pyo.Binary)  # 1 if teacher busy in that day-group

# precompute teacher terms by day-group
teacher_day_terms = {}
for d in days:
    teacher_day_terms[d] = [
        (e, ts)
        for (e, ts) in m1.X_index
        if (teacher_id in as_list(event_to_teachers.get(e)))
        and (time_to_day[ts] == d)
    ]

M_day = {d: len(teacher_day_terms[d]) for d in days}

# if teacher has any event in day d => teacher_busy_day[d] = 1
def cbd_link_upper_rule(m, d):
    terms = teacher_day_terms[d]
    if not terms:
        return m.teacher_busy_day[d] == 0
    return sum(m.x[e, ts] for (e, ts) in terms) <= M_day[d] * m.teacher_busy_day[d]

def cbd_link_lower_rule(m, d):
    terms = teacher_day_terms[d]
    if not terms:
        return pyo.Constraint.Skip
    return m.teacher_busy_day[d] <= sum(m.x[e, ts] for (e, ts) in terms)

m1.CBD_LinkUpper = pyo.Constraint(m1.CBD_DAYS, rule=cbd_link_upper_rule)
m1.CBD_LinkLower = pyo.Constraint(m1.CBD_DAYS, rule=cbd_link_lower_rule)

# hard bounds: min <= number of busy day-groups <= max
m1.CBD_MinBusy = pyo.Constraint(expr=sum(m1.teacher_busy_day[d] for d in m1.CBD_DAYS) >= min_busy)
m1.CBD_MaxBusy = pyo.Constraint(expr=sum(m1.teacher_busy_day[d] for d in m1.CBD_DAYS) <= max_busy)


In [ ]:

# rerun-safe cleanup
if hasattr(m1, "PreferResourceGroupCapacity"):
    m1.del_component(m1.PreferResourceGroupCapacity)

m1.PreferResourceGroupCapacity = pyo.ConstraintList()

# prefer-resource constraints only
pref_cids = set(
    df_constraints.loc[
        df_constraints["constraint_type"] == "PreferResourcesConstraint",
        "constraint_id"
    ]
)

# helper maps
# constraint -> events it applies to
cid_to_events = (
    df_constraint_applied.loc[
        (df_constraint_applied["constraint_id"].isin(pref_cids)) &
        (df_constraint_applied["path"] == "AppliesTo/Events/Event"),
        ["constraint_id", "reference"]
    ]
    .groupby("constraint_id")["reference"]
    .apply(lambda s: set(s.astype(str).str.strip()))
    .to_dict()
)

# Dont think this is needed
eg_to_events = (
    event_groups.groupby("event_group_id")["event_id"]
    .apply(lambda s: set(s.astype(str).str.strip()))
    .to_dict()
)


# constraint -> preferred rooms (explicit resources in constraint_params)
# IMPORTANT: if your df_constraint_params was previously merged and duplicated,
# drop_duplicates keeps it safe.
cp = df_constraint_params[["constraint_id", "path", "value"]].drop_duplicates().copy()

cid_to_pref_rooms = (
    cp.loc[
        (cp["constraint_id"].isin(pref_cids)) &
        (cp["path"] == "Resources/Resource"),
        ["constraint_id", "value"]
    ]
    .groupby("constraint_id")["value"]
    .apply(lambda s: set(s.astype(str).str.strip()))
    .to_dict()
)

for cid in pref_cids:
    applies_events = set(cid_to_events.get(cid, set()))
    pref_rooms = set(cid_to_pref_rooms.get(cid, set()))
    n_rooms = len(pref_rooms)

    if not applies_events or n_rooms == 0:
        continue

    for t in m1.T:
        terms = [m1.x[e, ts] for (e, ts) in occupies_at_time.get(t, []) if e in applies_events]
        if terms:
            m1.PreferResourceGroupCapacity.add(sum(terms) <= n_rooms)



### Solve

In [ ]:
# ---- Objective ---- (hard constraints only)
m1.obj = pyo.Objective(expr=0, sense=pyo.minimize)

In [ ]:
# ---- Solve ---

solver = pyo.SolverFactory("gurobi")
solver.options["TimeLimit"] = 1000
# solver.options["MIPGap"] = 0.0

res = solver.solve(m1, tee=True)

print(res.solver.termination_condition) 

In [ ]:
start_x = {}
for idx in m1.X_index:
    v = pyo.value(m1.x[idx], exception=False)
    start_x[idx] = 0.0 if v is None else float(v)

#start_x

### 2c: Model 2

### LimitIdleTimes


In [ ]:
# Build day -> timeslots once
day_to_times = defaultdict(list)
for t in m1.T:
    day_to_times[time_to_day[t]].append(t)

pos_in_day = {}
for d, times in day_to_times.items():
    for i, t in enumerate(times, start=1):
        pos_in_day[(d, t)] = i

In [ ]:
# LimitIdleStudents

# q_student[s,t] = 1 if student s is busy at time t
student_terms = defaultdict(list)  # (student, t) -> [(e,ts),...]
for t in T:
    for (e, ts) in occupies_at_time.get(t, []):
        for s in as_list(event_to_students.get(e)):
            if s in m1.S:
                student_terms[(s, t)].append((e, ts))

# busy indicator per time
m1.q_student = pyo.Var(m1.S, m1.T, domain=pyo.Binary)

def student_busy_def_rule(m, s, t):
    return m.q_student[s, t] == sum(m.x[e, ts] for (e, ts) in student_terms.get((s, t), []))
m1.StudentBusyDef = pyo.Constraint(m1.S, m1.T, rule=student_busy_def_rule)

# day-level vars
m1.y_student = pyo.Var(m1.S, m1.DAYS, domain=pyo.Binary)                 # busy at least once in day
m1.first_student = pyo.Var(m1.S, m1.DAYS, domain=pyo.NonNegativeIntegers)
m1.last_student  = pyo.Var(m1.S, m1.DAYS, domain=pyo.NonNegativeIntegers)
m1.idle_student  = pyo.Var(m1.S, m1.DAYS, domain=pyo.NonNegativeIntegers)

# busy count per day
def student_busy_count_rule(m, s, d):
    return sum(m.q_student[s, t] for t in day_to_times[d])
m1.student_busy_count = pyo.Expression(m1.S, m1.DAYS, rule=student_busy_count_rule)

# y_student links
def student_present_up_rule(m, s, d):
    nd = len(day_to_times[d])
    return m.student_busy_count[s, d] <= nd * m.y_student[s, d]
m1.StudentPresentUp = pyo.Constraint(m1.S, m1.DAYS, rule=student_present_up_rule)

def student_present_lo_rule(m, s, d):
    return m.student_busy_count[s, d] >= m.y_student[s, d]
m1.StudentPresentLo = pyo.Constraint(m1.S, m1.DAYS, rule=student_present_lo_rule)

# first/last fixes
def student_first_fix_rule(m, s, d):
    nd = len(day_to_times[d])
    return m.first_student[s, d] <= 1 + (nd - 1) * m.y_student[s, d]
m1.StudentFirstFix = pyo.Constraint(m1.S, m1.DAYS, rule=student_first_fix_rule)

def student_first_lb_rule(m, s, d):
    return m.first_student[s, d] >= 1
m1.StudentFirstLB = pyo.Constraint(m1.S, m1.DAYS, rule=student_first_lb_rule)

def student_last_fix_rule(m, s, d):
    nd = len(day_to_times[d])
    return m.last_student[s, d] <= nd * m.y_student[s, d]
m1.StudentLastFix = pyo.Constraint(m1.S, m1.DAYS, rule=student_last_fix_rule)

# envelopes for first/last busy positions
m1.StudentFirstEnv = pyo.ConstraintList()
m1.StudentLastEnv = pyo.ConstraintList()
for s in m1.S:
    for d in m1.DAYS:
        nd = len(day_to_times[d])
        for t in day_to_times[d]:
            p = pos_in_day[(d, t)]
            m1.StudentFirstEnv.add(
                m1.first_student[s, d] <= p + nd * (1 - m1.q_student[s, t])
            )
            m1.StudentLastEnv.add(
                m1.last_student[s, d] >= p * m1.q_student[s, t]
            )

# idle count per day
def student_idle_def_rule(m, s, d):
    return m.idle_student[s, d] == (
        m.last_student[s, d] - m.first_student[s, d] + 1 - m.student_busy_count[s, d]
    )
m1.StudentIdleDef = pyo.Constraint(m1.S, m1.DAYS, rule=student_idle_def_rule)

# soft linear penalty, weight=7, min=max=0 => deviation is idle count
m1.PenaltyIdleStudents = pyo.Expression(
    expr=7 * sum(m1.idle_student[s, d] for s in m1.S for d in m1.DAYS)
)


In [ ]:
# LimitIdleTeachers

# q_teacher[tr,t] = 1 if teacher tr is busy at time t
teacher_terms = defaultdict(list)  # (teacher, t) -> [(e,ts),...]
for t in T:
    for (e, ts) in occupies_at_time.get(t, []):
        for tr in as_list(event_to_teachers.get(e)):
            if tr in m1.Teachers:
                teacher_terms[(tr, t)].append((e, ts))

# busy indicator per time
m1.q_teacher = pyo.Var(m1.Teachers, m1.T, domain=pyo.Binary)

def teacher_busy_def_rule(m, tr, t):
    return m.q_teacher[tr, t] == sum(m.x[e, ts] for (e, ts) in teacher_terms.get((tr, t), []))
m1.TeacherBusyDef = pyo.Constraint(m1.Teachers, m1.T, rule=teacher_busy_def_rule)

# day-level vars
m1.y_teacher = pyo.Var(m1.Teachers, m1.DAYS, domain=pyo.Binary)  # busy at least once in day
m1.first_teacher = pyo.Var(m1.Teachers, m1.DAYS, domain=pyo.NonNegativeIntegers)
m1.last_teacher  = pyo.Var(m1.Teachers, m1.DAYS, domain=pyo.NonNegativeIntegers)
m1.idle_teacher  = pyo.Var(m1.Teachers, m1.DAYS, domain=pyo.NonNegativeIntegers)

# busy count per day
def teacher_busy_count_rule(m, tr, d):
    return sum(m.q_teacher[tr, t] for t in day_to_times[d])
m1.teacher_busy_count = pyo.Expression(m1.Teachers, m1.DAYS, rule=teacher_busy_count_rule)

# y_teacher links
def teacher_present_up_rule(m, tr, d):
    nd = len(day_to_times[d])
    return m.teacher_busy_count[tr, d] <= nd * m.y_teacher[tr, d]
m1.TeacherPresentUp = pyo.Constraint(m1.Teachers, m1.DAYS, rule=teacher_present_up_rule)

def teacher_present_lo_rule(m, tr, d):
    return m.teacher_busy_count[tr, d] >= m.y_teacher[tr, d]
m1.TeacherPresentLo = pyo.Constraint(m1.Teachers, m1.DAYS, rule=teacher_present_lo_rule)

# first/last fixes
def teacher_first_fix_rule(m, tr, d):
    nd = len(day_to_times[d])
    return m.first_teacher[tr, d] <= 1 + (nd - 1) * m.y_teacher[tr, d]
m1.TeacherFirstFix = pyo.Constraint(m1.Teachers, m1.DAYS, rule=teacher_first_fix_rule)

def teacher_first_lb_rule(m, tr, d):
    return m.first_teacher[tr, d] >= 1
m1.TeacherFirstLB = pyo.Constraint(m1.Teachers, m1.DAYS, rule=teacher_first_lb_rule)

def teacher_last_fix_rule(m, tr, d):
    nd = len(day_to_times[d])
    return m.last_teacher[tr, d] <= nd * m.y_teacher[tr, d]
m1.TeacherLastFix = pyo.Constraint(m1.Teachers, m1.DAYS, rule=teacher_last_fix_rule)

# envelopes for first/last busy positions
m1.TeacherFirstEnv = pyo.ConstraintList()
m1.TeacherLastEnv = pyo.ConstraintList()
for tr in m1.Teachers:
    for d in m1.DAYS:
        nd = len(day_to_times[d])
        for t in day_to_times[d]:
            p = pos_in_day[(d, t)]
            m1.TeacherFirstEnv.add(
                m1.first_teacher[tr, d] <= p + nd * (1 - m1.q_teacher[tr, t])
            )
            m1.TeacherLastEnv.add(
                m1.last_teacher[tr, d] >= p * m1.q_teacher[tr, t]
            )

# idle count per day
def teacher_idle_def_rule(m, tr, d):
    return m.idle_teacher[tr, d] == (
        m.last_teacher[tr, d] - m.first_teacher[tr, d] + 1 - m.teacher_busy_count[tr, d]
    )
m1.TeacherIdleDef = pyo.Constraint(m1.Teachers, m1.DAYS, rule=teacher_idle_def_rule)

# soft linear penalty, weight=7
m1.PenaltyIdleTeachers = pyo.Expression(
    expr=7 * sum(m1.idle_teacher[tr, d] for tr in m1.Teachers for d in m1.DAYS)
)


### Spread Event Constraints

In [ ]:
# SpreadEventsNeighbordays
df_time_groups= pd.read_csv(base / "time_group_membership.csv")

cid = "SpreadEventsNeighbordays"

# 1) weight (soft)
w = 8

# 2) AppliesTo course/event groups
applied_course_groups = (
    df_constraint_applied.loc[
        (df_constraint_applied["constraint_id"] == cid) &
        (df_constraint_applied["path"] == "AppliesTo/EventGroups/EventGroup"),
        "reference"
    ]
    .dropna().astype(str).str.strip().unique().tolist()
)

# event_group_id -> events
cg_to_events = event_groups.groupby("event_group_id")["event_id"].apply(set).to_dict()

# 3) TimeGroup -> (min,max) from constraint_params
cp = df_constraint_params.loc[
    df_constraint_params["constraint_id"] == cid, ["path", "value"]
].reset_index(drop=True)

tg_bounds = {}   # tg -> (min,max)
cur_tg = None
cur_min = 0
cur_max = 10**9

for r in cp.itertuples(index=False):
    if r.path == "TimeGroups/TimeGroup":
        if cur_tg is not None:
            tg_bounds[cur_tg] = (int(cur_min), int(cur_max))
        cur_tg = str(r.value).strip()
        cur_min, cur_max = 0, 10**9
    elif r.path == "TimeGroups/TimeGroup/Minimum":
        cur_min = int(r.value)
    elif r.path == "TimeGroups/TimeGroup/Maximum":
        cur_max = int(r.value)

if cur_tg is not None:
    tg_bounds[cur_tg] = (int(cur_min), int(cur_max))

neighbor_tgs = list(tg_bounds.keys())

# 4) TimeGroup -> member times (start times that count)
#    Requires df_time_group_membership with columns group_id,time_id
tg_to_times = (
    df_time_groups.groupby("group_id")["time_id"].apply(set).to_dict()
)

# 5) Build index pairs (course_group, time_group) that are relevant
idx_pairs = []
for cg in applied_course_groups:
    if cg not in cg_to_events:
        continue
    for tg in neighbor_tgs:
        idx_pairs.append((cg, tg))

m1.SpreadNbrIdx = pyo.Set(dimen=2, initialize=idx_pairs)
m1.spread_nbr_under = pyo.Var(m1.SpreadNbrIdx, domain=pyo.NonNegativeReals)
m1.spread_nbr_over  = pyo.Var(m1.SpreadNbrIdx, domain=pyo.NonNegativeReals)

# helper expression: number of starts from course-group cg in time-group tg
def starts_count_expr(cg, tg):
    evs = cg_to_events.get(cg, set())
    times_in_tg = tg_to_times.get(tg, set())
    return sum(
        m1.x[e, ts]
        for (e, ts) in m1.X_index
        if (e in evs) and (ts in times_in_tg)
    )

# 6) deviation linearization
def spread_under_rule(m, cg, tg):
    mn, mx = tg_bounds[tg]
    return m.spread_nbr_under[cg, tg] >= mn - starts_count_expr(cg, tg)

def spread_over_rule(m, cg, tg):
    mn, mx = tg_bounds[tg]
    return m.spread_nbr_over[cg, tg] >= starts_count_expr(cg, tg) - mx

m1.SpreadNbrUnderConstr = pyo.Constraint(m1.SpreadNbrIdx, rule=spread_under_rule)
m1.SpreadNbrOverConstr  = pyo.Constraint(m1.SpreadNbrIdx, rule=spread_over_rule)

# 7) soft penalty expression (linear)
m1.SpreadNeighborPenalty = pyo.Expression(
    expr=w * sum(
        m1.spread_nbr_under[cg, tg] + m1.spread_nbr_over[cg, tg]
        for (cg, tg) in m1.SpreadNbrIdx
    )
)

In [ ]:
# SpreadEventsNeighbordays
df_time_groups= pd.read_csv(base / "time_group_membership.csv")

cid = "ClassWeekStability1"

# 1) weight (soft)
w = 8

# 2) AppliesTo course/event groups
applied_course_groups = (
    df_constraint_applied.loc[
        (df_constraint_applied["constraint_id"] == cid) &
        (df_constraint_applied["path"] == "AppliesTo/EventGroups/EventGroup"),
        "reference"
    ]
    .dropna().astype(str).str.strip().unique().tolist()
)

# event_group_id -> events
cg_to_events = event_groups.groupby("event_group_id")["event_id"].apply(set).to_dict()

# 3) TimeGroup -> (min,max) from constraint_params
cp = df_constraint_params.loc[
    df_constraint_params["constraint_id"] == cid, ["path", "value"]
].reset_index(drop=True)

tg_bounds = {}   # tg -> (min,max)
cur_tg = None
cur_min = 0
cur_max = 10**9

for r in cp.itertuples(index=False):
    if r.path == "TimeGroups/TimeGroup":
        if cur_tg is not None:
            tg_bounds[cur_tg] = (int(cur_min), int(cur_max))
        cur_tg = str(r.value).strip()
        cur_min, cur_max = 0, 10**9
    elif r.path == "TimeGroups/TimeGroup/Minimum":
        cur_min = int(r.value)
    elif r.path == "TimeGroups/TimeGroup/Maximum":
        cur_max = int(r.value)

if cur_tg is not None:
    tg_bounds[cur_tg] = (int(cur_min), int(cur_max))

neighbor_tgs = list(tg_bounds.keys())

# 4) TimeGroup -> member times (start times that count)
#    Requires df_time_group_membership with columns group_id,time_id
tg_to_times = (
    df_time_groups.groupby("group_id")["time_id"].apply(set).to_dict()
)

# 5) Build index pairs (course_group, time_group) that are relevant
idx_pairs = []
for cg in applied_course_groups:
    if cg not in cg_to_events:
        continue
    for tg in neighbor_tgs:
        idx_pairs.append((cg, tg))

m1.SpreadCW1Idx = pyo.Set(dimen=2, initialize=idx_pairs)
m1.spread_CW1_under = pyo.Var(m1.SpreadCW1Idx, domain=pyo.NonNegativeReals)
m1.spread_CW1_over  = pyo.Var(m1.SpreadCW1Idx, domain=pyo.NonNegativeReals)

# helper expression: number of starts from course-group cg in time-group tg
def starts_count_expr(cg, tg):
    evs = cg_to_events.get(cg, set())
    times_in_tg = tg_to_times.get(tg, set())
    return sum(
        m1.x[e, ts]
        for (e, ts) in m1.X_index
        if (e in evs) and (ts in times_in_tg)
    )

# 6) deviation linearization
def spread_under_rule(m, cg, tg):
    mn, mx = tg_bounds[tg]
    return m.spread_CW1_under[cg, tg] >= mn - starts_count_expr(cg, tg)

def spread_over_rule(m, cg, tg):
    mn, mx = tg_bounds[tg]
    return m.spread_CW1_over[cg, tg] >= starts_count_expr(cg, tg) - mx

m1.SpreadCW1UnderConstr = pyo.Constraint(m1.SpreadCW1Idx, rule=spread_under_rule)
m1.SpreadCW1OverConstr  = pyo.Constraint(m1.SpreadCW1Idx, rule=spread_over_rule)

# 7) soft penalty expression (linear)
m1.SpreadNeighborPenaltyCW1 = pyo.Expression(
    expr=w * sum(
        m1.spread_CW1_under[cg, tg] + m1.spread_CW1_over[cg, tg]
        for (cg, tg) in m1.SpreadCW1Idx
    )
)



In [ ]:
# ClassWeekStability2

for cid in ["ClassWeekStability2", "ClassWeekStability3", "ClassWeekStability4", "ClassWeekStability5"]:

    tag = cid[-1]  # "2","3","4","5"

    # cleanup if rerun
    for nm in [
        f"SpreadCW{tag}Idx",
        f"spread_CW{tag}_under",
        f"spread_CW{tag}_over",
        f"SpreadCW{tag}UnderConstr",
        f"SpreadCW{tag}OverConstr",
        f"SpreadNeighborPenaltyCW{tag}",
    ]:
        if hasattr(m1, nm):
            m1.del_component(getattr(m1, nm))

    # weight
    w = float(
        df_constraints.loc[df_constraints["constraint_id"] == cid, "weight"].iloc[0]
    )

    # applies-to event groups
    applied_course_groups = (
        df_constraint_applied.loc[
            (df_constraint_applied["constraint_id"] == cid) &
            (df_constraint_applied["path"] == "AppliesTo/EventGroups/EventGroup"),
            "reference"
        ]
        .dropna().astype(str).str.strip().unique().tolist()
    )

    # params for this constraint (dedup to avoid merged duplicates)
    cp = (
        df_constraint_params.loc[
            df_constraint_params["constraint_id"] == cid, ["path", "value"]
        ]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    # parse timegroup bounds
    tg_bounds = {}
    cur_tg, cur_min, cur_max = None, 0, 10**9

    for r in cp.itertuples(index=False):
        if r.path == "TimeGroups/TimeGroup":
            if cur_tg is not None:
                tg_bounds[cur_tg] = (int(cur_min), int(cur_max))
            cur_tg = str(r.value).strip()
            cur_min, cur_max = 0, 10**9
        elif r.path == "TimeGroups/TimeGroup/Minimum":
            cur_min = int(r.value)
        elif r.path == "TimeGroups/TimeGroup/Maximum":
            cur_max = int(r.value)

    if cur_tg is not None:
        tg_bounds[cur_tg] = (int(cur_min), int(cur_max))

    tgs = list(tg_bounds.keys())

    # index pairs
    idx_pairs = []
    for cg in applied_course_groups:
        if cg not in cg_to_events:
            continue
        for tg in tgs:
            idx_pairs.append((cg, tg))

    # sets/vars
    setattr(m1, f"SpreadCW{tag}Idx", pyo.Set(dimen=2, initialize=idx_pairs))
    idxset = getattr(m1, f"SpreadCW{tag}Idx")

    setattr(m1, f"spread_CW{tag}_under", pyo.Var(idxset, domain=pyo.NonNegativeReals))
    setattr(m1, f"spread_CW{tag}_over",  pyo.Var(idxset, domain=pyo.NonNegativeReals))

    under = getattr(m1, f"spread_CW{tag}_under")
    over  = getattr(m1, f"spread_CW{tag}_over")

    # helper
    def starts_count_expr(cg, tg):
        evs = cg_to_events.get(cg, set())
        times_in_tg = tg_to_times.get(tg, set())
        return sum(
            m1.x[e, ts]
            for (e, ts) in m1.X_index
            if (e in evs) and (ts in times_in_tg)
        )

    # constraints
    def under_rule(m, cg, tg):
        mn, mx = tg_bounds[tg]
        return under[cg, tg] >= mn - starts_count_expr(cg, tg)

    def over_rule(m, cg, tg):
        mn, mx = tg_bounds[tg]
        return over[cg, tg] >= starts_count_expr(cg, tg) - mx

    setattr(m1, f"SpreadCW{tag}UnderConstr", pyo.Constraint(idxset, rule=under_rule))
    setattr(m1, f"SpreadCW{tag}OverConstr",  pyo.Constraint(idxset, rule=over_rule))

    # penalty expression
    setattr(
        m1,
        f"SpreadNeighborPenaltyCW{tag}",
        pyo.Expression(expr=w * sum(under[cg, tg] + over[cg, tg] for (cg, tg) in idxset))
    )


### ClusterBusyTimes

In [ ]:
# ClusterBusyDaysOccupiedPenalty

# day groups used by this constraint
cid = "ClusterBusyDaysOccupiedPenalty"

# busy-day indicator for each teacher and day-group
m1.teacher_busy_day_occ = pyo.Var(m1.Teachers, m1.DAYS, domain=pyo.Binary)

# precompute teacher/day feasible starts
teacher_day_terms_occ = {}
for tr in m1.Teachers:
    for d in m1.DAYS:
        teacher_day_terms_occ[(tr, d)] = [
            (e, ts)
            for (e, ts) in m1.X_index
            if (tr in as_list(event_to_teachers.get(e))) and (time_to_day[ts] == d)
        ]

M_day_occ = {(tr, d): len(teacher_day_terms_occ[(tr, d)]) for tr in m1.Teachers for d in m1.DAYS}

# link busy-day binary to actual schedule
def cbdop_link_up_rule(m, tr, d):
    terms = teacher_day_terms_occ[(tr, d)]
    if not terms:
        return m.teacher_busy_day_occ[tr, d] == 0
    return sum(m.x[e, ts] for (e, ts) in terms) <= M_day_occ[(tr, d)] * m.teacher_busy_day_occ[tr, d]

def cbdop_link_lo_rule(m, tr, d):
    terms = teacher_day_terms_occ[(tr, d)]
    if not terms:
        return pyo.Constraint.Skip
    return m.teacher_busy_day_occ[tr, d] <= sum(m.x[e, ts] for (e, ts) in terms)

m1.CBDOP_LinkUp = pyo.Constraint(m1.Teachers, m1.DAYS, rule=cbdop_link_up_rule)
m1.CBDOP_LinkLo = pyo.Constraint(m1.Teachers, m1.DAYS, rule=cbdop_link_lo_rule)

# min=0, max=0 => deviation per teacher is number of busy days in these groups
# soft linear weight=1
m1.PenaltyClusterBusyDaysOccupiedTeachers = pyo.Expression(
    expr=sum(m1.teacher_busy_day_occ[tr, d] for tr in m1.Teachers for d in m1.DAYS)
)


In [ ]:
# ClusterBusyDaysOffStudents

# busy-day indicator for each student and day
m1.student_busy_day = pyo.Var(m1.S, m1.DAYS, domain=pyo.Binary)

# precompute student/day feasible starts
student_day_terms = {}
for s in m1.S:
    for d in m1.DAYS:
        student_day_terms[(s, d)] = [
            (e, ts)
            for (e, ts) in m1.X_index
            if (s in as_list(event_to_students.get(e))) and (time_to_day[ts] == d)
        ]

M_day_student = {(s, d): len(student_day_terms[(s, d)]) for s in m1.S for d in m1.DAYS}

# link busy-day binary to schedule
def student_busy_day_up_rule(m, s, d):
    terms = student_day_terms[(s, d)]
    if not terms:
        return m.student_busy_day[s, d] == 0
    return sum(m.x[e, ts] for (e, ts) in terms) <= M_day_student[(s, d)] * m.student_busy_day[s, d]

def student_busy_day_lo_rule(m, s, d):
    terms = student_day_terms[(s, d)]
    if not terms:
        return pyo.Constraint.Skip
    return m.student_busy_day[s, d] <= sum(m.x[e, ts] for (e, ts) in terms)

m1.StudentBusyDayUp = pyo.Constraint(m1.S, m1.DAYS, rule=student_busy_day_up_rule)
m1.StudentBusyDayLo = pyo.Constraint(m1.S, m1.DAYS, rule=student_busy_day_lo_rule)

# min=10, max=10
min_busy = 10
max_busy = 10

m1.cluster_student_under = pyo.Var(m1.S, domain=pyo.NonNegativeReals)
m1.cluster_student_over  = pyo.Var(m1.S, domain=pyo.NonNegativeReals)

def cluster_student_under_rule(m, s):
    return m.cluster_student_under[s] >= min_busy - sum(m.student_busy_day[s, d] for d in m.DAYS)

def cluster_student_over_rule(m, s):
    return m.cluster_student_over[s] >= sum(m.student_busy_day[s, d] for d in m.DAYS) - max_busy

m1.ClusterStudentUnder = pyo.Constraint(m1.S, rule=cluster_student_under_rule)
m1.ClusterStudentOver  = pyo.Constraint(m1.S, rule=cluster_student_over_rule)

# soft linear weight=1
m1.PenaltyClusterBusyDaysOffStudents = pyo.Expression(
    expr=sum(m1.cluster_student_under[s] + m1.cluster_student_over[s] for s in m1.S)
)


In [ ]:
# LimitBusyMoreThanOne

cid = "LimitBusyMoreThanOne"

# cleanup on rerun
for nm in [
    "LBT_RES","LBT_DAYS","busy_count_lbt","active_lbt",
    "lbt_under","lbt_over","LBTActiveUp","LBTActiveLo",
    "LBTUnderDef","LBTOverDef","PenaltyLimitBusyMoreThanOne"
]:
    if hasattr(m1, nm):
        m1.del_component(getattr(m1, nm))

# params from constraint table
w = 4
min_busy = 2
max_busy = 5

# day groups used by this constraint
#lbt_days = (
#    df_constraint_params.loc[
#        (df_constraint_params["constraint_id"] == cid) &
#        (df_constraint_params["path"] == "TimeGroups/TimeGroup"),
#        "value"
#    ]
#    .dropna().astype(str).str.strip().tolist()
#)

# ---- FORCE SCOPE: all teachers only ----
teacher_ids = sorted(set(all_teachers))
m1.LBT_RES = pyo.Set(initialize=teacher_ids)
#m1.LBT_DAYS = pyo.Set(initialize=lbt_days)

# teacher -> events
teacher_to_events = (
    event_resources.loc[event_resources["role"] == "Teacher"]
    .groupby("reference_resource_id")["event_id"]
    .apply(set).to_dict()
)

# busy-count terms by (teacher, day)
lbt_terms = {}
for tr in m1.LBT_RES:
    evs = teacher_to_events.get(tr, set())
    for d in m1.DAYS:
        terms = []
        for t in day_to_times[d]:
            for (e, ts) in occupies_at_time.get(t, []):
                if e in evs:
                    terms.append((e, ts))
        lbt_terms[(tr, d)] = terms

# number of busy periods for teacher tr on day d
m1.busy_count_lbt = pyo.Expression(
    m1.LBT_RES, m1.DAYS,
    rule=lambda m, tr, d: sum(m.x[e, ts] for (e, ts) in lbt_terms[(tr, d)])
)

# active indicator (teacher has >=1 busy period that day)
m1.active_lbt = pyo.Var(m1.LBT_RES, m1.DAYS, domain=pyo.Binary)

def lbt_active_up_rule(m, tr, d):
    nd = len(day_to_times[d])  # big-M
    return m.busy_count_lbt[tr, d] <= nd * m.active_lbt[tr, d]

def lbt_active_lo_rule(m, tr, d):
    return m.busy_count_lbt[tr, d] >= m.active_lbt[tr, d]

m1.LBTActiveUp = pyo.Constraint(m1.LBT_RES, m1.DAYS, rule=lbt_active_up_rule)
m1.LBTActiveLo = pyo.Constraint(m1.LBT_RES, m1.DAYS, rule=lbt_active_lo_rule)

# deviation vars (linear)
m1.lbt_under = pyo.Var(m1.LBT_RES, m1.DAYS, domain=pyo.NonNegativeReals)
m1.lbt_over  = pyo.Var(m1.LBT_RES, m1.DAYS, domain=pyo.NonNegativeReals)

# special case busy_count=0 => no min shortfall penalty
def lbt_under_rule(m, tr, d):
    M = min_busy
    return m.lbt_under[tr, d] >= min_busy - m.busy_count_lbt[tr, d] - M * (1 - m.active_lbt[tr, d])

def lbt_over_rule(m, tr, d):
    return m.lbt_over[tr, d] >= m.busy_count_lbt[tr, d] - max_busy

m1.LBTUnderDef = pyo.Constraint(m1.LBT_RES, m1.DAYS, rule=lbt_under_rule)
m1.LBTOverDef  = pyo.Constraint(m1.LBT_RES, m1.DAYS, rule=lbt_over_rule)

# soft penalty
m1.PenaltyLimitBusyMoreThanOne = pyo.Expression(
    expr=w * sum(m1.lbt_under[tr, d] + m1.lbt_over[tr, d] for tr in m1.LBT_RES for d in m1.DAYS)
)


### Results

In [ ]:
# ---- Objective ---- (with soft constraints)

if hasattr(m1, "obj"):
    m1.del_component(m1.obj)


penalty_total = (
    m1.PenaltyIdleStudents +
    m1.PenaltyIdleTeachers +
    m1.SpreadNeighborPenalty +
    m1.SpreadNeighborPenaltyCW1 +
    m1.SpreadNeighborPenaltyCW2 +
    m1.SpreadNeighborPenaltyCW3 +
    m1.SpreadNeighborPenaltyCW4 +
    m1.SpreadNeighborPenaltyCW5 +
    m1.PenaltyClusterBusyDaysOccupiedTeachers +
    m1.PenaltyClusterBusyDaysOffStudents +
    m1.PenaltyLimitBusyMoreThanOne
)

m1.obj = pyo.Objective(expr=penalty_total, sense=pyo.minimize)


In [ ]:
for idx in m1.X_index:
    m1.x[idx].value = start_x[idx]


solver2 = pyo.SolverFactory("gurobi")
solver2.options["TimeLimit"] = 200
solver2.options["MIPFocus"] = 1
solver2.options["Heuristics"] = 1  #0.4
solver2.options["NoRelHeurTime"] = 200
solver2.options["RINS"] = 20



# warmstart=True is important
res2 = solver2.solve(m1, tee=True, warmstart=True)

tc2 = res2.solver.termination_condition
print("Phase 2 termination:", tc2)
print("Phase 2 objective:", pyo.value(m1.obj))


In [ ]:
for idx in m1.X_index:
    m1.x[idx].value = start_x[idx]


solver2 = pyo.SolverFactory("gurobi")
solver2.options["TimeLimit"] = 20000
solver2.options["MIPFocus"] = 1
#solver2.options["Heuristics"] = 0.4
#solver2.options["NoRelHeurTime"] = 1000
solver2.options["RINS"] = 20



# warmstart=True is important
res2 = solver2.solve(m1, tee=True, warmstart=True)

tc2 = res2.solver.termination_condition
print("Phase 2 termination:", tc2)
print("Phase 2 objective:", pyo.value(m1.obj))


In [ ]:
# Build chosen_start from m1 solution
chosen_start = {}
for e in m1.E:
    picked = [ts for ts in feasible_starts[e] if pyo.value(m1.x[e, ts]) > 0.5]
    if len(picked) != 1:
        raise ValueError(f"Event {e}: expected exactly 1 chosen start, got {picked}")
    chosen_start[e] = picked[0]

print("Assigned starts:", len(chosen_start), "out of", len(list(m1.E)))
#chosen_start

### 2d: Model 3

In [ ]:
# ---- Model ----
m2 = pyo.ConcreteModel()

# ---- Sets ----
m2.E = pyo.Set(initialize=E) # Set of events
m2.T = pyo.Set(initialize=T, ordered=True) # Set of timeslots (ordered)
m2.R = pyo.Set(initialize=R) # Set of rooms

# standard set up
m2.X_index = pyo.Set(
    dimen=2,
    initialize=[(e, r) for e in E for r in R]
)



events_occupying_t = defaultdict(list)
for e in E:
    ts = chosen_start[e]
    for t in occ_times[(e, ts)]:
        events_occupying_t[t].append(e)

# ---- Decision Variables ----
# Start-time decision variables:
# x[e, ts, r] = 1 if event e starts at time ts in room r
# Only define variables for feasible starts (and all rooms) (THIS SPEEDS UP MODEL)
m2.x = pyo.Var(m2.X_index, domain=pyo.Binary)

In [ ]:
# Standard set up

# 1) Each event gets exactly one room
def one_room_rule(m, e):
    return sum(m.x[e, r] for r in R) == 1
m2.OneRoom = pyo.Constraint(m2.E, rule=one_room_rule)


# 2) No room conflicts: at most one event can occupy a room at a time
def room_conflict_rule(m, r, t):
    terms = [m.x[e, r] for e in events_occupying_t.get(t, []) if (e, r) in m.X_index]
    if not terms:
        return pyo.Constraint.Skip
    return sum(terms) <= 1

m2.RoomConflict = pyo.Constraint(m2.R, m2.T, rule=room_conflict_rule)

### Results

In [ ]:
# ---- Objective ---- 
m2.obj = pyo.Objective(expr=0, sense=pyo.minimize)

In [ ]:
# ---- Solve ---

solver = pyo.SolverFactory("gurobi")
# solver.options["TimeLimit"] = 1000
# solver.options["MIPGap"] = 0.0

res = solver.solve(m2, tee=True)

print(res.solver.termination_condition) 